# M3 Hadoop MapReduce — Google Colab Edition

Run **Steps 1–12 in order** in Google Colab. In Step 2, upload both `mapper2.py` and `reducer.py` from your Mac. This notebook runs a local Python simulation of the MapReduce streaming process; no Hadoop cluster is required. The Student Practice section is preserved.

## Step 1 — Import libraries


In [1]:
from pathlib import Path
import subprocess
import sys
import shutil


## Step 2 — Set up Google Colab and upload mapper/reducer scripts

When prompted, select **both** `mapper2.py` and `reducer.py` from your Mac’s `Codebook` folder. Repeat this upload after a Colab runtime reset. No Mac-specific absolute paths are used.

In [2]:
# Step 2 — Mount Google Drive and use the real UCI course folder.
from pathlib import Path
import os
from google.colab import drive

drive.mount('/content/drive')

PROJECT_ROOT = (Path('/content/drive/MyDrive/Colab Notebooks/UCI')
                / 'UCI_425.07_Big_Data_Analytics' / '03_In-Class Practice')
ASSIGNMENTS_DIR = PROJECT_ROOT
CODEBOOK_DIR = PROJECT_ROOT / 'Codebook'
DATASET_DIR = PROJECT_ROOT / 'Dataset'
OUTPUTS_DIR = PROJECT_ROOT / 'Outputs'

print('Notebook working directory:', os.getcwd())
print('Google Drive course folder:', PROJECT_ROOT)
if not PROJECT_ROOT.is_dir():
    raise FileNotFoundError(
        f'Course folder not found: {PROJECT_ROOT}\n'
        'Verify the folder spelling in the mounted Drive. '
        'If the folder is in a shared drive or shortcut, its path may differ.'
    )

# Prefer scripts in the course Codebook; if absent, check the previously
# identified local-project Codebook only when it is also synced to Drive.
SCRIPT_FOLDERS = [CODEBOOK_DIR,
    Path('/content/drive/MyDrive/Colab Notebooks/UCI/UCI_Hadoop_Apache_Airflow/03_In-Class Practice/Codebook')]

def locate_script(name):
    for folder in SCRIPT_FOLDERS:
        path = folder / name
        if path.is_file():
            return path
    return CODEBOOK_DIR / name

MAPPER = locate_script('mapper2.py')
REDUCER = locate_script('reducer.py')
INPUT_FILE = DATASET_DIR / 'wordcount_input.txt'
OUTPUT_FILE = OUTPUTS_DIR / 'wordcount_output.txt'

print('Codebook folder exists:', CODEBOOK_DIR.is_dir())
print('Mapper found:', MAPPER.is_file(), MAPPER)
print('Reducer found:', REDUCER.is_file(), REDUCER)
if not MAPPER.is_file() or not REDUCER.is_file():
    raise FileNotFoundError(
        'mapper2.py and/or reducer.py are not in the Google Drive Codebook folder.\n'
        f'Expected: {CODEBOOK_DIR}\n'
        'Upload or copy the two scripts to that folder in Google Drive, '
        'then rerun this cell. The Mac /Users/... path is not accessible to Colab.'
    )


Mounted at /content/drive
Notebook working directory: /content
Google Drive course folder: /content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/03_In-Class Practice
Codebook folder exists: True
Mapper found: True /content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/03_In-Class Practice/Codebook/mapper2.py
Reducer found: True /content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/03_In-Class Practice/Codebook/reducer.py


## Step 3 — Prepare Dataset and Outputs folders

The Colab directories are created automatically in Step 2.

In [3]:
DATASET_DIR.mkdir(parents=True, exist_ok=True)
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
print("Dataset directory:", DATASET_DIR)
print("Outputs directory:", OUTPUTS_DIR)

Dataset directory: /content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/03_In-Class Practice/Dataset
Outputs directory: /content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/03_In-Class Practice/Outputs


## Step 4 — Create sample input file if it does not exist


In [4]:
if not INPUT_FILE.exists():
    INPUT_FILE.write_text(
        """Hadoop is useful for big data.

Hadoop MapReduce uses mapper and reducer files.

This assignment runs mapper2.py and reducer.py.
""",
        encoding="utf-8",
    )

    print("Created sample input file:", INPUT_FILE)

else:
    print("Using existing input file:", INPUT_FILE)


Created sample input file: /content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/03_In-Class Practice/Dataset/wordcount_input.txt


## Step 5 — Check mapper and reducer files


In [5]:
# Step 5 — Verify scripts from the mounted Google Drive folder.
for label, script in [('Mapper', MAPPER), ('Reducer', REDUCER)]:
    print(f'{label}: {"FOUND" if script.is_file() else "MISSING"} — {script}')
if not all(p.is_file() for p in (MAPPER, REDUCER)):
    raise FileNotFoundError('Missing scripts in mounted Google Drive. Check Step 2.')


Mapper: FOUND — /content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/03_In-Class Practice/Codebook/mapper2.py
Reducer: FOUND — /content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/03_In-Class Practice/Codebook/reducer.py


## Step 6 — Clean previous output


In [6]:
if OUTPUT_FILE.exists():
    OUTPUT_FILE.unlink()
    print("Previous output removed.")
else:
    print("No previous output file found.")


No previous output file found.


## Step 7 — Define helper function to run a command


In [7]:
def run_command(command, input_text: str, timeout_seconds: int = 30):
    return subprocess.run(
        command,
        input=input_text,
        text=True,
        capture_output=True,
        check=False,
        timeout=timeout_seconds,
    )


## Step 8 — Run mapper2.py


In [8]:
input_text = INPUT_FILE.read_text(
    encoding="utf-8",
    errors="ignore"
)

mapper_result = run_command(
    [sys.executable, str(MAPPER)],
    input_text=input_text
)

if mapper_result.returncode != 0:
    raise RuntimeError(
        "Mapper failed:\n"
        + mapper_result.stderr
    )

print("Mapper ran successfully.")
print("\nFirst mapper output lines:\n")

print(
    "\n".join(
        mapper_result.stdout.splitlines()[:20]
    )
)


Mapper ran successfully.

First mapper output lines:

hadoop	1
useful	1
big	1
data	1
hadoop	1
mapreduce	1
uses	1
mapper	1
reducer	1
files	1
assignment	1
runs	1
mapper2	1
py	1
reducer	1
py	1


## Step 9 — Sort mapper output

Hadoop normally sorts mapper output before sending it to the reducer. Since this notebook runs locally, we sort it here.


In [9]:
# Sort mapper records and retain the final newline for streaming compatibility.
sorted_mapper_output = "\n".join(sorted(mapper_result.stdout.splitlines())) + "\n"
print("Sorted mapper output sample:\n")
print("\n".join(sorted_mapper_output.splitlines()[:20]))


Sorted mapper output sample:

assignment	1
big	1
data	1
files	1
hadoop	1
hadoop	1
mapper	1
mapper2	1
mapreduce	1
py	1
py	1
reducer	1
reducer	1
runs	1
useful	1
uses	1


## Step 10 — Run reducer.py


In [10]:
reducer_result = run_command(
    [sys.executable, str(REDUCER)],
    input_text=sorted_mapper_output
)

if reducer_result.returncode != 0:
    raise RuntimeError(
        "Reducer failed:\n"
        + reducer_result.stderr
    )

print("Reducer ran successfully.")
print("\nFirst reducer output lines:\n")

print(
    "\n".join(
        reducer_result.stdout.splitlines()[:20]
    )
)


Reducer ran successfully.

First reducer output lines:

assignment	1
big	1
data	1
files	1
hadoop	2
mapper	1
mapper2	1
mapreduce	1
py	2
reducer	2
runs	1
useful	1
uses	1


## Step 11 — Save output file


In [11]:
OUTPUT_FILE.write_text(reducer_result.stdout, encoding="utf-8")

print("Output saved to:")
print(OUTPUT_FILE)


Output saved to:
/content/drive/MyDrive/Colab Notebooks/UCI/UCI_425.07_Big_Data_Analytics/03_In-Class Practice/Outputs/wordcount_output.txt


## Step 12 — Display final WordCount output


In [12]:
print("================ WORD COUNT OUTPUT ================")
print(OUTPUT_FILE.read_text(encoding="utf-8", errors="ignore"))


================ WORD COUNT OUTPUT ================
assignment	1
big	1
data	1
files	1
hadoop	2
mapper	1
mapper2	1
mapreduce	1
py	2
reducer	2
runs	1
useful	1
uses	1



# Student Practice Section

The following cells are added **after the full working code**.

Students can practice by editing the input text, running mapper, sorting mapper output, running reducer, and saving/displaying the output.

Instructor note: the answer code is included below. You can comment it out or hide it the same way you did for Day 2.


## Practice 1 — Student updates the input text

Students should replace the text below with their own 5 or more lines.


In [13]:
# ================= STUDENT PRACTICE 1 =================
# TODO: Replace the text below with your own paragraph.

practice_text = """
Hadoop processes large data files.
MapReduce separates work into map and reduce steps.
The mapper creates word and count pairs.
The reducer adds counts for each word.
Python can simulate MapReduce logic locally.
"""

# PRACTICE_INPUT_FILE = (
#     DATASET_DIR / "student_practice_input.txt"
# )

# PRACTICE_INPUT_FILE.write_text(
#     practice_text.strip() + "\n",
#     encoding="utf-8"
# )

# print("Practice input file created:")
# print(PRACTICE_INPUT_FILE)

# print("\nPractice input text:\n")

# print(
#     PRACTICE_INPUT_FILE.read_text(
#         encoding="utf-8"
#     )
# )


## Practice 2 — Student runs mapper2.py on practice input


In [14]:
# ================= STUDENT PRACTICE 2 =================
# TODO: Uncomment the practice input creation in Practice 1 before running.
if "PRACTICE_INPUT_FILE" not in globals() or not PRACTICE_INPUT_FILE.is_file():
    print("Complete Practice 1 first: uncomment its file creation lines.")
else:
    practice_input_text = PRACTICE_INPUT_FILE.read_text(encoding="utf-8", errors="ignore")
    practice_mapper_result = run_command([sys.executable, str(MAPPER)], practice_input_text)
    if practice_mapper_result.returncode != 0:
        raise RuntimeError("Practice mapper failed:\n" + practice_mapper_result.stderr)
    print("Practice mapper output sample:\n")
    print("\n".join(practice_mapper_result.stdout.splitlines()[:30]))


Complete Practice 1 first: uncomment its file creation lines.


## Practice 3 — Student sorts mapper output


In [15]:
# ================= STUDENT PRACTICE 3 =================
# TODO: Run Practice 2 before this cell.
if "practice_mapper_result" not in globals():
    print("Run Practice 2 first.")
else:
    practice_sorted_mapper_output = "\n".join(sorted(practice_mapper_result.stdout.splitlines())) + "\n"
    print("Practice sorted mapper output sample:\n")
    print("\n".join(practice_sorted_mapper_output.splitlines()[:30]))


Run Practice 2 first.


## Practice 4 — Student runs reducer.py


In [16]:
# ================= STUDENT PRACTICE 4 =================
# TODO: Send sorted mapper output into reducer.py.

# practice_reducer_result = run_command(
#     [sys.executable, str(REDUCER)],
#     input_text=practice_sorted_mapper_output
# )

# if practice_reducer_result.returncode != 0:
#     raise RuntimeError(
#         "Practice reducer failed:\n"
#         + practice_reducer_result.stderr
#     )

# print("Practice reducer output:\n")

# print(
#     practice_reducer_result.stdout
# )


## Practice 5 — Student saves practice output


In [17]:
# ================= STUDENT PRACTICE 5 =================
# TODO: Save reducer output to a practice output file.

# PRACTICE_OUTPUT_FILE = OUTPUTS_DIR / "m3_student_practice_output.txt"
# PRACTICE_OUTPUT_FILE.write_text(practice_reducer_result.stdout, encoding="utf-8")

# print("Practice output saved to:")
# print(PRACTICE_OUTPUT_FILE)


## Practice 6 — Student displays practice output


In [18]:
# ================= STUDENT PRACTICE 6 =================
# TODO: Display the saved practice output file.

# print("================ STUDENT PRACTICE OUTPUT ================")
# print(PRACTICE_OUTPUT_FILE.read_text(encoding="utf-8", errors="ignore"))


# Optional Challenge Practice

Students can try to modify the input text and answer:

1. Which word appears most often?
2. How many unique words are in the output?
3. What happens if punctuation or uppercase letters are added?


In [19]:
# ================= OPTIONAL CHALLENGE =================
# Students may uncomment and run this after Practice 6.

# lines = PRACTICE_OUTPUT_FILE.read_text(encoding="utf-8").strip().splitlines()
# word_counts = []

# for line in lines:
#     word, count = line.split("	")
#     word_counts.append((word, int(count)))

# most_common_word = max(word_counts, key=lambda item: item[1])
# unique_word_count = len(word_counts)

# print("Most common word:", most_common_word)
# print("Number of unique words:", unique_word_count)
